<!-- project-navigation -->
# 13_RandomForest
Role: Model experiments and analysis.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    recall_score,
    f1_score
)

# ============================================================
# CHARGEMENT DES DATASETS ENCODÉS
# ============================================================

train_encoded = pd.read_csv(
    "data/processed/train_encoded.csv"
)

test_encoded = pd.read_csv(
    "data/processed/test_encoded.csv"
)

print("==========================================")
print("DATASETS ENCODÉS")
print("==========================================")

print("TRAIN :", train_encoded.shape)
print("TEST  :", test_encoded.shape)

print("\nNaN TRAIN :", train_encoded.isna().sum().sum())
print("NaN TEST  :", test_encoded.isna().sum().sum())

# ============================================================
# SÉPARATION X / y
# ============================================================

X = train_encoded.drop(
    columns=["TARGET", "SK_ID_CURR"],
    errors="ignore"
)

y = train_encoded["TARGET"]

print("\nX :", X.shape)
print("y :", y.shape)

# ============================================================
# CONTRÔLE FINAL
# ============================================================

print("\nVariables non numériques :")

print(
    "TRAIN :",
    train_encoded.select_dtypes(exclude=np.number).columns.tolist()
)

print(
    "TEST :",
    test_encoded.select_dtypes(exclude=np.number).columns.tolist()
)

print("\nNombre de features :", X.shape[1])

DATASETS ENCODÉS
TRAIN : (307511, 242)
TEST  : (48744, 242)

NaN TRAIN : 0
NaN TEST  : 0

X : (307511, 240)
y : (307511,)

Variables non numériques :
TRAIN : []
TEST : []

Nombre de features : 240


In [2]:
# ============================================================
# 1. SÉPARATION TRAIN / VALIDATION
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

X_train : (246008, 240)
X_valid : (61503, 240)
y_train : (246008,)
y_valid : (61503,)


In [3]:
# ============================================================
# 2. RANDOM FOREST
# ============================================================

model_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

print("Random Forest entraîné.")

Random Forest entraîné.


In [4]:
# ============================================================
# 3. PRÉDICTIONS
# ============================================================

y_pred = model_rf.predict(X_valid)
y_proba = model_rf.predict_proba(X_valid)[:, 1]

# ============================================================
# 4. ÉVALUATION
# ============================================================

print("==========================================")
print("RÉSULTATS RANDOM FOREST")
print("==========================================")

print("\nAccuracy :", accuracy_score(y_valid, y_pred))

print("\nRecall défaut :",
      recall_score(y_valid, y_pred))

print("\nF1-score défaut :",
      f1_score(y_valid, y_pred))

print("\nROC-AUC :",
      roc_auc_score(y_valid, y_proba))

print("\nMatrice de confusion :")
print(confusion_matrix(y_valid, y_pred))

print("\nClassification Report :")
print(classification_report(y_valid, y_pred))

RÉSULTATS RANDOM FOREST

Accuracy : 0.9183617059330439

Recall défaut : 0.04169184290030212

F1-score défaut : 0.07617295308187673

ROC-AUC : 0.766951231402726

Matrice de confusion :
[[56275   263]
 [ 4758   207]]

Classification Report :
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.44      0.04      0.08      4965

    accuracy                           0.92     61503
   macro avg       0.68      0.52      0.52     61503
weighted avg       0.88      0.92      0.89     61503



In [5]:
# ============================================================
# FEATURE IMPORTANCE - RANDOM FOREST
# ============================================================

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("==========================================")
print("TOP 20 FEATURES - RANDOM FOREST")
print("==========================================")

print(
    feature_importance.head(20).to_string(index=False)
)

TOP 20 FEATURES - RANDOM FOREST
                  Feature  Importance
             EXT_SOURCE_2    0.048276
             EXT_SOURCE_3    0.042584
             EXT_SOURCE_1    0.024619
           AVG_CREDIT_AGE    0.021738
            DAYS_EMPLOYED    0.021450
               DAYS_BIRTH    0.018953
          DAYS_ID_PUBLISH    0.017702
              AMT_ANNUITY    0.017639
   DAYS_LAST_PHONE_CHANGE    0.017514
        DAYS_REGISTRATION    0.017246
               AMT_CREDIT    0.017120
           AVG_DEBT_RATIO    0.016864
         AVG_DOWN_PAYMENT    0.016729
AVG_CNT_INSTALMENT_FUTURE    0.015981
          AMT_GOODS_PRICE    0.015670
              AVG_ANNUITY    0.015542
        AVG_PAYMENT_DELAY    0.015541
     TOTAL_PAYMENT_AMOUNT    0.015374
      AVG_CREDIT_AMOUNT_x    0.015018
 TOTAL_INSTALLMENT_AMOUNT    0.015017


In [6]:
# ============================================================
# EXPORT TOP 20 FEATURE IMPORTANCE VERS EXCEL
# ============================================================

import pandas as pd

# Top 20
top20_excel = feature_importance.head(20).copy()

# Ajouter le rang
top20_excel.insert(
    0,
    "Rang",
    range(1, len(top20_excel) + 1)
)

# Ajouter l'interprétation métier
interpretations = {
    "EXT_SOURCE_2":
        "Score externe de solvabilité, très informatif sur le risque de défaut",

    "EXT_SOURCE_3":
        "Score externe de solvabilité, fortement lié au risque de défaut",

    "EXT_SOURCE_1":
        "Score externe de solvabilité supplémentaire",

    "AVG_CREDIT_AGE":
        "Ancienneté moyenne des crédits historiques",

    "DAYS_EMPLOYED":
        "Ancienneté de l'emploi du client",

    "DAYS_BIRTH":
        "Âge du client, exprimé en nombre de jours depuis la naissance",

    "DAYS_ID_PUBLISH":
        "Ancienneté depuis la publication de la pièce d'identité",

    "AMT_ANNUITY":
        "Montant de l'annuité du crédit actuel",

    "DAYS_LAST_PHONE_CHANGE":
        "Nombre de jours depuis le dernier changement de téléphone",

    "DAYS_REGISTRATION":
        "Ancienneté de l'enregistrement du client",

    "AMT_CREDIT":
        "Montant du crédit demandé",

    "AVG_DEBT_RATIO":
        "Ratio moyen d'endettement issu de l'historique de crédit",

    "AVG_DOWN_PAYMENT":
        "Montant moyen de l'apport sur les crédits historiques",

    "AVG_CNT_INSTALMENT_FUTURE":
        "Nombre moyen d'échéances futures",

    "AMT_GOODS_PRICE":
        "Prix du bien financé",

    "AVG_ANNUITY":
        "Montant moyen des annuités des crédits historiques",

    "AVG_PAYMENT_DELAY":
        "Retard moyen de paiement observé dans l'historique",

    "TOTAL_PAYMENT_AMOUNT":
        "Montant total payé dans l'historique des paiements",

    "AVG_CREDIT_AMOUNT_x":
        "Montant moyen des crédits historiques",

    "TOTAL_INSTALLMENT_AMOUNT":
        "Montant total des échéances historiques"
}

top20_excel["Interprétation métier"] = (
    top20_excel["Feature"].map(interpretations)
)

# Renommer la colonne
top20_excel = top20_excel.rename(
    columns={
        "Feature": "Variable",
        "Importance": "Importance Random Forest"
    }
)

# Arrondir l'importance
top20_excel["Importance Random Forest"] = (
    top20_excel["Importance Random Forest"].round(6)
)

# ============================================================
# SAUVEGARDE EXCEL
# ============================================================

output_path = "data/processed/random_forest_top20_features.xlsx"

top20_excel.to_excel(
    output_path,
    index=False
)

print("==========================================")
print("EXPORT EXCEL TERMINÉ")
print("==========================================")
print("Fichier :", output_path)

print("\nTop 20 exporté :")
print(top20_excel.to_string(index=False))

EXPORT EXCEL TERMINÉ
Fichier : data/processed/random_forest_top20_features.xlsx

Top 20 exporté :
 Rang                  Variable  Importance Random Forest                                                 Interprétation métier
    1              EXT_SOURCE_2                  0.048276 Score externe de solvabilité, très informatif sur le risque de défaut
    2              EXT_SOURCE_3                  0.042584       Score externe de solvabilité, fortement lié au risque de défaut
    3              EXT_SOURCE_1                  0.024619                           Score externe de solvabilité supplémentaire
    4            AVG_CREDIT_AGE                  0.021738                            Ancienneté moyenne des crédits historiques
    5             DAYS_EMPLOYED                  0.021450                                      Ancienneté de l'emploi du client
    6                DAYS_BIRTH                  0.018953         Âge du client, exprimé en nombre de jours depuis la naissance
    7 

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# ============================================================
# RANDOMIZED SEARCH — VERSION OPTIMISÉE POUR LA RAM
# ============================================================

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=1
)

param_distributions = {
    "n_estimators": [100, 150, 200],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced", "balanced_subsample"]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=8,              # au lieu de 15
    scoring="roc_auc",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1               # IMPORTANT : éviter la saturation RAM
)

print("==========================================")
print("DÉBUT RANDOMIZED SEARCH")
print("==========================================")

random_search.fit(X_train, y_train)

print("\n==========================================")
print("RECHERCHE TERMINÉE")
print("==========================================")

print("Meilleurs paramètres :")
print(random_search.best_params_)

print("\nMeilleur ROC-AUC CV :")
print(random_search.best_score_)

DÉBUT RANDOMIZED SEARCH
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END class_weight=balanced_subsample, max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=2, n_estimators=150; total time= 1.5min
[CV] END class_weight=balanced_subsample, max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=2, n_estimators=150; total time= 1.5min
[CV] END class_weight=balanced_subsample, max_depth=None, max_features=log2, min_samples_leaf=5, min_samples_split=2, n_estimators=150; total time=19.1min
[CV] END class_weight=balanced, max_depth=15, max_features=sqrt, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time= 1.3min
[CV] END class_weight=balanced, max_depth=15, max_features=sqrt, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time= 1.2min
[CV] END class_weight=balanced, max_depth=15, max_features=sqrt, min_samples_leaf=5, min_samples_split=5, n_estimators=100; total time= 1.7min
[CV] END class_weight=

In [8]:
# ============================================================
# MEILLEUR MODÈLE
# ============================================================

model_rf_optimized = random_search.best_estimator_

print("==========================================")
print("RANDOM FOREST OPTIMISÉ")
print("==========================================")

print(model_rf_optimized)

RANDOM FOREST OPTIMISÉ
RandomForestClassifier(class_weight='balanced', min_samples_leaf=5, n_jobs=1,
                       random_state=42)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    recall_score,
    f1_score
)

# ============================================================
# RANDOM FOREST OPTIMISÉ
# ============================================================

rf_optimized = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features="sqrt",
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("==========================================")
print("ENTRAÎNEMENT RANDOM FOREST OPTIMISÉ")
print("==========================================")

rf_optimized.fit(X_train, y_train)

print("Entraînement terminé.")

ENTRAÎNEMENT RANDOM FOREST OPTIMISÉ
Entraînement terminé.


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_optimized = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features="sqrt",
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("==========================================")
print("ENTRAÎNEMENT RANDOM FOREST OPTIMISÉ")
print("==========================================")

rf_optimized.fit(X_train, y_train)

print("Entraînement terminé.")

ENTRAÎNEMENT RANDOM FOREST OPTIMISÉ
Entraînement terminé.


In [11]:
# ============================================================
# PRÉDICTIONS DU RANDOM FOREST OPTIMISÉ
# ============================================================

y_pred_optimized = rf_optimized.predict(X_valid)

y_proba_optimized = rf_optimized.predict_proba(
    X_valid
)[:, 1]

print("Prédictions terminées.")

Prédictions terminées.


In [12]:
# ============================================================
# ÉVALUATION
# ============================================================

accuracy_optimized = accuracy_score(
    y_valid,
    y_pred_optimized
)

recall_optimized = recall_score(
    y_valid,
    y_pred_optimized
)

f1_optimized = f1_score(
    y_valid,
    y_pred_optimized
)

roc_auc_optimized = roc_auc_score(
    y_valid,
    y_proba_optimized
)

print("==========================================")
print("RANDOM FOREST OPTIMISÉ")
print("==========================================")

print(f"Accuracy      : {accuracy_optimized:.4f}")
print(f"Recall défaut : {recall_optimized:.4f}")
print(f"F1-score      : {f1_optimized:.4f}")
print(f"ROC-AUC       : {roc_auc_optimized:.4f}")

print("\nMatrice de confusion :")
print(confusion_matrix(y_valid, y_pred_optimized))

print("\nClassification Report :")
print(classification_report(y_valid, y_pred_optimized))

RANDOM FOREST OPTIMISÉ
Accuracy      : 0.8948
Recall défaut : 0.2564
F1-score      : 0.2823
ROC-AUC       : 0.7664

Matrice de confusion :
[[53758  2780]
 [ 3692  1273]]

Classification Report :
              precision    recall  f1-score   support

           0       0.94      0.95      0.94     56538
           1       0.31      0.26      0.28      4965

    accuracy                           0.89     61503
   macro avg       0.62      0.60      0.61     61503
weighted avg       0.89      0.89      0.89     61503



In [13]:
# ============================================================
# OPTIMISATION DU SEUIL DE DÉCISION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.61, 0.05)

results_threshold = []

for threshold in thresholds:

    # Nouvelle prédiction selon le seuil
    y_pred_threshold = (
        y_proba_optimized >= threshold
    ).astype(int)

    results_threshold.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_valid,
            y_pred_threshold
        ),
        "Precision_defaut": precision_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        ),
        "Recall_defaut": recall_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        ),
        "F1_defaut": f1_score(
            y_valid,
            y_pred_threshold,
            zero_division=0
        )
    })

results_threshold = pd.DataFrame(
    results_threshold
)

print("==========================================")
print("RÉSULTATS PAR SEUIL")
print("==========================================")

display(
    results_threshold.round(4)
)

RÉSULTATS PAR SEUIL


,Threshold,Accuracy,Precision_defaut,Recall_defaut,F1_defaut
0,0.10,0.1663,0.0872,0.9855,0.1603
1,0.15,0.3038,0.1000,0.9525,0.1809
2,0.20,0.4550,0.1176,0.8842,0.2076
3,0.25,0.5905,0.1421,0.8089,0.2418
4,0.30,0.6976,0.1701,0.7076,0.2742
5,0.35,0.7778,0.2027,0.5972,0.3026
6,0.40,0.8334,0.2345,0.4697,0.3128
7,0.45,0.8708,0.2714,0.3561,0.3080
8,0.50,0.8948,0.3141,0.2564,0.2823
9,0.55,0.9085,0.3610,0.1734,0.2343


In [14]:
# ============================================================
# MEILLEUR SEUIL SELON LE F1-SCORE
# ============================================================

best_threshold_row = results_threshold.loc[
    results_threshold["F1_defaut"].idxmax()
]

print("==========================================")
print("MEILLEUR SEUIL")
print("==========================================")

print(
    "Seuil :",
    best_threshold_row["Threshold"]
)

print(
    "Accuracy :",
    round(best_threshold_row["Accuracy"], 4)
)

print(
    "Precision défaut :",
    round(best_threshold_row["Precision_defaut"], 4)
)

print(
    "Recall défaut :",
    round(best_threshold_row["Recall_defaut"], 4)
)

print(
    "F1-score défaut :",
    round(best_threshold_row["F1_defaut"], 4)
)

MEILLEUR SEUIL
Seuil : 0.40000000000000013
Accuracy : 0.8334
Precision défaut : 0.2345
Recall défaut : 0.4697
F1-score défaut : 0.3128


In [15]:
# ============================================================
# COMPARAISON SEUIL 0.50 VS SEUIL OPTIMISÉ
# ============================================================

comparison = pd.DataFrame({
    "Métrique": [
        "Accuracy",
        "Precision défaut",
        "Recall défaut",
        "F1-score défaut"
    ],

    "Seuil 0.50": [
        accuracy_score(y_valid, (y_proba_optimized >= 0.50).astype(int)),
        precision_score(y_valid, (y_proba_optimized >= 0.50).astype(int)),
        recall_score(y_valid, (y_proba_optimized >= 0.50).astype(int)),
        f1_score(y_valid, (y_proba_optimized >= 0.50).astype(int))
    ],

    "Seuil optimisé": [
        best_threshold_row["Accuracy"],
        best_threshold_row["Precision_defaut"],
        best_threshold_row["Recall_defaut"],
        best_threshold_row["F1_defaut"]
    ]
})

print("==========================================")
print("COMPARAISON")
print("==========================================")

display(comparison.round(4))

COMPARAISON


,Métrique,Seuil 0.50,Seuil optimisé
0,Accuracy,0.8948,0.8334
1,Precision défaut,0.3141,0.2345
2,Recall défaut,0.2564,0.4697
3,F1-score défaut,0.2823,0.3128


In [16]:
print("-- TRAIN --")
print("Nombre total :", len(y_train))
print("Sans défaut :", (y_train == 0).sum(), f"({(y_train == 0).mean():.2%})")
print("Défaut :", (y_train == 1).sum(), f"({(y_train == 1).mean():.2%})")

print("\n-- VALIDATION --")
print("Nombre total :", len(y_valid))
print("Sans défaut :", (y_valid == 0).sum(), f"({(y_valid == 0).mean():.2%})")
print("Défaut :", (y_valid == 1).sum(), f"({(y_valid == 1).mean():.2%})")

-- TRAIN --
Nombre total : 246008
Sans défaut : 226148 (91.93%)
Défaut : 19860 (8.07%)

-- VALIDATION --
Nombre total : 61503
Sans défaut : 56538 (91.93%)
Défaut : 4965 (8.07%)


In [17]:
print("Variables disponibles :")
for name in ["rf_model", "rf", "random_forest", "model_rf", "X_train", "X_valid", "y_train", "y_valid"]:
    if name in globals():
        print("✓", name, "->", type(globals()[name]))

Variables disponibles :
✓ rf -> <class 'sklearn.ensemble._forest.RandomForestClassifier'>
✓ model_rf -> <class 'sklearn.ensemble._forest.RandomForestClassifier'>
✓ X_train -> <class 'pandas.DataFrame'>
✓ X_valid -> <class 'pandas.DataFrame'>
✓ y_train -> <class 'pandas.Series'>
✓ y_valid -> <class 'pandas.Series'>


In [18]:
print("========== RANDOM FOREST OPTIMISÉ ==========")

print("Meilleurs paramètres :")
print(random_search.best_params_)

print("\nMeilleur ROC-AUC CV :")
print(random_search.best_score_)

========== RANDOM FOREST OPTIMISÉ ==========
Meilleurs paramètres :
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}

Meilleur ROC-AUC CV :
0.7597416132496099


In [19]:
print("X_valid :", X_valid.shape)
print("y_valid :", y_valid.shape)

X_valid : (61503, 240)
y_valid : (61503,)


In [21]:
from sklearn.ensemble import RandomForestClassifier

best_rf = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=2,
    min_samples_leaf=5,
    max_features="sqrt",
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Entraînement du Random Forest optimisé...")

best_rf.fit(X_train, y_train)

print("✓ Modèle entraîné.")

Entraînement du Random Forest optimisé...
✓ Modèle entraîné.


In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_pred_rf_opt = best_rf.predict(X_valid)
y_proba_rf_opt = best_rf.predict_proba(X_valid)[:, 1]

print("========== RANDOM FOREST — OPTIMISÉ ==========")

print(f"Accuracy      : {accuracy_score(y_valid, y_pred_rf_opt):.4f}")
print(f"Precision     : {precision_score(y_valid, y_pred_rf_opt):.4f}")
print(f"Recall défaut : {recall_score(y_valid, y_pred_rf_opt):.4f}")
print(f"F1-score      : {f1_score(y_valid, y_pred_rf_opt):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_valid, y_proba_rf_opt):.4f}")

print("\n========== MATRICE DE CONFUSION ==========")
print(confusion_matrix(y_valid, y_pred_rf_opt))

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(y_valid, y_pred_rf_opt))

========== RANDOM FOREST — OPTIMISÉ ==========
Accuracy      : 0.8948
Precision     : 0.3141
Recall défaut : 0.2564
F1-score      : 0.2823
ROC-AUC       : 0.7664

========== MATRICE DE CONFUSION ==========
[[53758  2780]
 [ 3692  1273]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0       0.94      0.95      0.94     56538
           1       0.31      0.26      0.28      4965

    accuracy                           0.89     61503
   macro avg       0.62      0.60      0.61     61503
weighted avg       0.89      0.89      0.89     61503

